# 01 - Data and features

Thin wrapper around the `ml_trading` package. All of the logic lives in
`src/ml_trading/` so that it is unit tested and runs identically on a laptop and
on a cluster; this notebook only wires it to Delta.

Load the price panel, build the stationary feature set and horizon-aware labels,
and persist both to Delta.


In [ ]:
%pip install -e . yfinance


In [ ]:
from ml_trading.config import ExperimentConfig
from ml_trading.data import coverage_report, load_prices
from ml_trading.features import build_dataset
from ml_trading.spark_io import FEATURES_TABLE, PRICES_TABLE, write_table

config = ExperimentConfig.from_yaml("configs/logistic_h21.yaml")
config.data


In [ ]:
prices = load_prices(config.data)
display(coverage_report(prices, config.data.symbols).head(20))


In [ ]:
# The tidy contract is one row per (date, symbol) with adjusted OHLCV.
write_table(spark, prices, PRICES_TABLE, partition_by="symbol")
prices.tail()


In [ ]:
dataset = build_dataset(prices, config.features, config.label)
print(f"{len(dataset):,} labelled rows, {dataset['symbol'].nunique()} symbols")
write_table(spark, dataset, FEATURES_TABLE, partition_by="symbol")
dataset.tail()


### What changed relative to the original notebook

* Features are returns, ratios, ranks and z-scores rather than raw price levels
  (`sma_20`, `std_20`), so a model fitted on 2015 prices is not extrapolating in 2020.
* `forward_return` is the log return over exactly `label.horizon` days, and the label
  is its sign *relative to the cross-section*, so the model learns relative value
  rather than the market direction every name shares.
* Nothing here shifts the label into the feature space: the execution lag is applied
  once, in the backtest.
